Neals funnel is a synthetic model but challenging for posteriors samplers due to its unusual geometry. It has exponential shape in one direction and a narrow funnel beding in the other direction. 


Mathematically it is defined as follows

$$
z \sim \mathcal{N}(0,3) \\
x \sim \mathcal{N}(0, e^{z/2})
$$

In [1]:
import pymc as pm
import pandas as pd
import numpy as np

c:\Users\mkami\OneDrive\Pulpit\UNI\posteriordb\.venv310\lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [8]:
CSV_R = "neals_draws_rstan/"
stan_points_centered = pd.read_csv(CSV_R + "eval_points_centered.csv")
stan_log_probs_centered = pd.read_csv(CSV_R + "log_probs_centered.csv")
stan_grads_centered = pd.read_csv(CSV_R + "grads_centered.csv")


stan_points_noncentered = pd.read_csv(CSV_R + "eval_points_noncentered.csv")
stan_log_probs_noncentered = pd.read_csv(CSV_R + "log_probs_noncentered.csv")
stan_grads_noncentered = pd.read_csv(CSV_R + "grads_noncentered.csv")

D = 9

In [3]:
with pm.Model() as noncentered_model:
    # Your existing PyMC model definition
    v = pm.Normal("v", mu=0, sigma=3)
    x_raw = pm.Normal("x_raw", mu=0, sigma=1, shape=9)
    x = pm.Deterministic("x", x_raw * pm.math.exp(v / 2))

    # 2. Compile the log-probability and gradient functions
    # This exposes the computational geometry used by the PyMC implementation
    logp_fn = noncentered_model.compile_logp()
    dlogp_fn = noncentered_model.compile_dlogp()


In [ ]:
x_raw_cols = [col for col in stan_points_noncentered.columns if 'x_raw' in col]
# 3. Evaluate PyMC at the fixed Stan points
pymc_log_probs = []
pymc_grads = []

# Iterate through the fixed set of representative evaluation points
for _, row in stan_points_noncentered.iterrows():
    # Map the Stan variables to PyMC variables. 
    # Stan's 'y' maps to PyMC's 'v', and Stan's 'x_raw' maps to PyMC's 'x_raw'
    point_dict = {
        "v": row["y"], 
        "x_raw": row[["x_raw[1]", "x_raw[2]", "x_raw[3]", "x_raw[4]", "x_raw[5]", "x_raw[6]", "x_raw[7]", "x_raw[8]", "x_raw[9]"]].values
    }
    
    # Evaluate the log density and gradients at the same points
    pymc_log_probs.append(logp_fn(point_dict))
    
    # PyMC returns gradients as a list/array in the order of the model's free parameters
    # Usually [grad_v, grad_x_raw]
    grads = dlogp_fn(point_dict)
    
    # Flatten the PyMC gradient output to match the Stan 1D array structure
    flat_grads = np.concatenate([np.atleast_1d(g) for g in grads])
    pymc_grads.append(flat_grads)



pymc_log_probs_noncentered = np.array(pymc_log_probs)
pymc_grads_noncentered = np.array(pymc_grads)

In [5]:
stan_log_probs_noncentered = np.array(stan_log_probs_noncentered)
stan_log_probs_flat = stan_log_probs_noncentered.flatten()

In [6]:
# 4. The Equivalence Protocol Verification
# Check that the difference in log densities is approximately constant in i[cite: 2]
logp_diff = stan_log_probs_flat - pymc_log_probs_noncentered
print("\n--- Implementation Equivalence Verification ---")
print(f"Log-density difference variance (should be ~0): {np.var(logp_diff):.6f}")

# Check that the difference in gradients is approximately zero[cite: 2]
# We calculate the mean absolute error between the gradient arrays
grad_diff = stan_grads_noncentered.values - pymc_grads_noncentered
print(f"Gradient difference max error (should be ~0): {np.max(np.abs(grad_diff)):.6f}")


--- Implementation Equivalence Verification ---
Log-density difference variance (should be ~0): 0.000000
Gradient difference max error (should be ~0): 0.000000


# centered model

In [7]:
with pm.Model() as centered_model:
    # Your existing PyMC model definition
    v = pm.Normal("v", mu=0, sigma=3)
    x = pm.Normal("x", mu=0, sigma=pm.math.exp(v / 2), shape=9)

    # 2. Compile the log-probability and gradient functions
    # This exposes the computational geometry used by the PyMC implementation
    logp_fn = centered_model.compile_logp()
    dlogp_fn = centered_model.compile_dlogp()

In [8]:
# 3. Evaluate PyMC at the fixed Stan points
pymc_log_probs = []
pymc_grads = []

for _, row in stan_points_centered.iterrows():
    # Map the Stan variables to PyMC variables. 
    # Stan's 'y' maps to PyMC's 'v', and Stan's 'x' maps to PyMC's 'x'
    point_dict = {
        "v": row["y"], 
        "x": row[["x[1]", "x[2]", "x[3]", "x[4]", "x[5]", "x[6]", "x[7]", "x[8]", "x[9]"]].values
    }
    
    # Evaluate the log density and gradients at the same points[cite: 2]
    pymc_log_probs.append(logp_fn(point_dict))
    
    # PyMC returns gradients as a list/array in the order of the model's free parameters
    # Usually [grad_v, grad_x]
    grads = dlogp_fn(point_dict)
    
    # Flatten the PyMC gradient output to match the Stan 1D array structure
    flat_grads = np.concatenate([np.atleast_1d(g) for g in grads])
    pymc_grads.append(flat_grads)

pymc_log_probs_centered = np.array(pymc_log_probs)
pymc_grads_centered = np.array(pymc_grads)

In [9]:
stan_log_probs_centered = np.array(stan_log_probs_centered)
stan_log_probs_centered_flat = stan_log_probs_centered.flatten()
# 4. The Equivalence Protocol Verification
# Check that the difference in log densities is approximately constant in i
logp_diff = stan_log_probs_centered_flat - pymc_log_probs_centered
print("\n--- Implementation Equivalence Verification ---")
print(f"Log-density difference variance (should be ~0): {np.var(logp_diff):.6f}")

# Check that the difference in gradients is approximately zero[cite: 2]
# We calculate the mean absolute error between the gradient arrays
grad_diff = stan_grads_centered.values - pymc_grads_centered
print(f"Gradient difference max error (should be ~0): {np.max(np.abs(grad_diff)):.6f}")


--- Implementation Equivalence Verification ---
Log-density difference variance (should be ~0): 0.000000
Gradient difference max error (should be ~0): 0.000000


In [ ]:
# sanity check
pymc_grads_centered.shape, stan_grads_centered.shape

((100, 10), (100, 10))

## beanmachine model implementation

In [53]:
import torch
import torch.distributions as dist
import beanmachine.ppl as bm

class NealsFunnelNonCentered:
    @bm.random_variable
    def v(self):
        # Wrap parameters in torch.tensor
        return dist.Normal(torch.tensor(0.0), torch.tensor(3.0))

    @bm.random_variable
    def x_raw(self):
        return dist.Normal(torch.zeros(9), torch.ones(9))

    @bm.functional
    def x(self):
        # Wrap the divisor in torch.tensor as well
        return self.x_raw() * torch.exp(self.v() / torch.tensor(2.0))

In [54]:
bm_model = NealsFunnelNonCentered()


In [55]:
import numpy as np
# 1. Define the underlying PyTorch distributions exactly as they are in the Bean Machine model
dist_v = dist.Normal(loc=0.0, scale=3.0)
dist_x_raw = dist.Normal(loc=torch.zeros(9), scale=torch.ones(9))

# 2. Evaluate at the fixed Stan points
bm_log_probs = []
bm_grads = []

# Iterate through the fixed set of representative evaluation points
# Iterate through the fixed set of representative evaluation points
for _, row in stan_points_noncentered.iterrows():
    
    # Cast the Pandas scalar to a standard Python float
    v_val = torch.tensor(float(row["y"]), dtype=torch.float32, requires_grad=True)
    
    # Extract the values, cast to a standard numpy float array, then to a tensor
    x_raw_array = row[["x_raw[1]", "x_raw[2]", "x_raw[3]", "x_raw[4]", "x_raw[5]", "x_raw[6]", "x_raw[7]", "x_raw[8]", "x_raw[9]"]].values.astype(float)
    x_raw_vals = torch.tensor(x_raw_array, dtype=torch.float32, requires_grad=True)

    # Calculate joint log probability using the PyTorch distributions
    logp_v = dist_v.log_prob(v_val)
    logp_x_raw = dist_x_raw.log_prob(x_raw_vals).sum()
    joint_logp = logp_v + logp_x_raw

    bm_log_probs.append(joint_logp.item())
    
    # Compute the gradients
    joint_logp.backward()
    
    # Flatten the PyTorch gradient output to match the Stan 1D array structure
    flat_grads = np.array(torch.cat([
        v_val.grad.unsqueeze(0), 
        x_raw_vals.grad
    ]).tolist())
    bm_grads.append(flat_grads)

bm_log_probs_noncentered = np.array(bm_log_probs)
bm_grads_noncentered = np.array(bm_grads)

In [56]:
# 3. The Equivalence Protocol Verification[cite: 1]
# Flatten the Stan log probabilities array
stan_log_probs_flat = np.array(stan_log_probs_noncentered).flatten()

# Check that the difference in log densities is approximately constant in i
logp_diff = stan_log_probs_flat - bm_log_probs_noncentered
print("\n--- Implementation Equivalence Verification ---")
print(f"Log-density difference variance (should be ~0): {np.var(logp_diff):.6f}")

# Check that the difference in gradients is approximately zero
# We calculate the mean absolute error between the gradient arrays
grad_diff = stan_grads_noncentered.values - bm_grads_noncentered
print(f"Gradient difference max error (should be ~0): {np.max(np.abs(grad_diff)):.6f}")


--- Implementation Equivalence Verification ---
Log-density difference variance (should be ~0): 0.000000
Gradient difference max error (should be ~0): 0.000000
